# Workshop 1: source profiling (Team 5, Extreme Weather Monitoring, Texas)

Sources: F1 NOAA Storm Events (historical impacts), F2 GHCN-Daily (historical station data),
F3 NWS alerts (incremental), F4 NWS station observations (incremental).
All inputs are read from `data/raw/` or `workshop_1/samples/`. Raw files are never modified.

In [1]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
STAGING = ROOT / "data" / "staging"
SAMPLES = ROOT / "workshop_1" / "samples"
USER_AGENT = "(extreme-weather-monitoring, team5-ud)"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)


def utc_stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")


def save_raw(r, path):
    """Save the response bytes and a .headers.json file next to them."""
    path.write_bytes(r.content)
    meta = {"url": r.url, "status": r.status_code, "retrieved_utc": utc_stamp(), "headers": dict(r.headers)}
    Path(str(path) + ".headers.json").write_text(json.dumps(meta, indent=1))
    return path

## 0. Station selection and crosswalk F2 -> F4

Step 1. GHCN-Daily station list (`S2_stations.txt`, fixed width). We keep first-order stations
(ID prefix `USW`) in Texas.

In [2]:
stations = pd.read_fwf(RAW / "S2_stations.txt", header=None,
                       colspecs=[(0, 11), (12, 20), (21, 30), (31, 37), (38, 40), (41, 71)],
                       names=["id", "lat", "lon", "elev_m", "state", "name"], dtype={"id": str, "state": str})
tx_usw = stations[stations["id"].str.startswith("USW") & (stations["state"] == "TX")]
print("All GHCN-Daily stations:", len(stations))
print("USW stations in TX:", len(tx_usw))

All GHCN-Daily stations: 133041
USW stations in TX: 171


Result: 171 first-order stations in Texas, the starting set.

Step 2. Inventory (`S2_inventory.txt`). A station stays only if it has TMAX, TMIN and PRCP, each
with first year <= 1991 and last year >= 2025. The inventory gives only first and last year, so gaps
inside the period are checked later on the daily data.

In [3]:
inventory = pd.read_csv(RAW / "S2_inventory.txt", sep=r"\s+", header=None,
                        names=["id", "lat", "lon", "element", "first_year", "last_year"], dtype={"id": str})
inv = inventory[inventory["id"].isin(tx_usw["id"]) & inventory["element"].isin(["TMAX", "TMIN", "PRCP"])]
ok = inv[(inv["first_year"] <= 1991) & (inv["last_year"] >= 2025)]
n_ok = ok.groupby("id")["element"].nunique()
long_ids = n_ok[n_ok == 3].index
selected = tx_usw[tx_usw["id"].isin(long_ids)].copy()   # semi-join
print("TX USW stations with TMAX, TMIN, PRCP from <=1991 to >=2025:", len(selected))
saved = pd.read_csv(STAGING / "tx_selected_stations.csv", dtype=str)
print("Same ids as data/staging/tx_selected_stations.csv:", set(saved["id"]) == set(selected["id"]))

TX USW stations with TMAX, TMIN, PRCP from <=1991 to >=2025: 48
Same ids as data/staging/tx_selected_stations.csv: True


Result: 48 stations have the three elements over the whole 1991-2025 period. This is the same list David saved in `data/staging/tx_selected_stations.csv`.

Step 3. Crosswalk to ICAO. The last 5 digits of a USW id are the WBAN number. We join with the
NOAA ISD station history, read as strings to keep leading zeros. ISD uses `99999` (WBAN) and
`999999` (USAF) as missing values, and one WBAN can have several history rows, so we keep the
latest row per WBAN (sorted by END).

In [4]:
isd = pd.read_csv(RAW / "isd-history.csv", dtype=str)
isd["WBAN"] = isd["WBAN"].replace("99999", pd.NA)
isd["USAF"] = isd["USAF"].replace("999999", pd.NA)
print("ISD rows:", len(isd), "| rows with WBAN sentinel 99999:", isd["WBAN"].isna().sum())

isd_icao = isd.dropna(subset=["WBAN", "ICAO"])
dup_wban = isd_icao["WBAN"].duplicated(keep=False)
print("ISD rows with ICAO whose WBAN appears more than once:", dup_wban.sum())
isd_latest = isd_icao.sort_values("END").drop_duplicates("WBAN", keep="last")

selected["wban"] = selected["id"].str[-5:]
cross = selected.merge(isd_latest[["WBAN", "ICAO", "END"]], left_on="wban", right_on="WBAN", how="left", indicator=True)
print(cross["_merge"].value_counts().to_dict())
print("Stations without ICAO:", cross.loc[cross["_merge"] == "left_only", ["id", "name"]].values.tolist())
cross = cross[cross["_merge"] == "both"].rename(columns={"ICAO": "icao"})

ISD rows: 29661 | rows with WBAN sentinel 99999: 25367
ISD rows with ICAO whose WBAN appears more than once: 1618
{'both': 47, 'left_only': 1, 'right_only': 0}
Stations without ICAO: [['USW00003951', 'LONGVIEW WSMO']]


Result: 47 of the 48 stations get an ICAO code. LONGVIEW WSMO (USW00003951) has no ICAO in ISD, so
it cannot be linked to F4 and is dropped. The left join is what lets us report it instead of losing it silently.

Step 4. The ICAO code must exist in the NWS stations catalogue for Texas. The catalogue is paged with
a cursor and the last `next` link returns an empty page, so we read pages until `features` is empty.
The catalogue also gives the forecast zone, county zone and time zone of each station.

In [5]:
nws_rows = []
pages = sorted(RAW.glob("nws_stations_tx_*_p*.json"), key=lambda p: int(p.stem.split("_p")[-1]))
for page in pages:
    features = json.loads(page.read_text())["features"]
    print(page.name, "features:", len(features))
    if not features:
        break
    for f in features:
        p = f["properties"]
        nws_rows.append({"icao": p["stationIdentifier"],
                         "forecast_zone": (p.get("forecast") or "").rsplit("/", 1)[-1],
                         "county_zone": (p.get("county") or "").rsplit("/", 1)[-1],
                         "timezone": p.get("timeZone")})
nws = pd.DataFrame(nws_rows)
print("NWS stations in the TX catalogue:", len(nws))

final = cross.merge(nws, on="icao", how="inner")
print("Stations with ICAO but not published by the NWS API:", sorted(set(cross["icao"]) - set(final["icao"])))
shared = final[final["icao"].duplicated(keep=False)]
print("ICAO codes shared by more than one GHCN id:")
display(shared[["id", "name", "wban", "icao", "END"]])
# Keep the GHCN id whose ISD history row is the most recent (the other one is an old WBAN of the same airport).
final = final.sort_values("END").drop_duplicates("icao", keep="last")
final = final[["id", "wban", "icao", "name", "lat", "lon", "forecast_zone", "county_zone", "timezone"]]
final = final.rename(columns={"id": "ghcn_id"}).sort_values("ghcn_id").reset_index(drop=True)

nws_stations_tx_20261116_005156_p0.json features: 500
nws_stations_tx_20261116_005157_p1.json features: 500
nws_stations_tx_20261116_005159_p2.json features: 500
nws_stations_tx_20261116_005200_p3.json features: 500
nws_stations_tx_20261116_005204_p4.json features: 500
nws_stations_tx_20261116_005205_p5.json features: 500
nws_stations_tx_20261116_005207_p6.json features: 500
nws_stations_tx_20261116_005208_p7.json features: 351
nws_stations_tx_20261116_005209_p8.json features: 0
NWS stations in the TX catalogue: 3851
Stations with ICAO but not published by the NWS API: []
ICAO codes shared by more than one GHCN id:


,id,name,wban,icao,END
29,USW00013961,FT WORTH MEACHAM FLD,13961,KFTW,20250827
44,USW00093904,FT WORTH MEACHAM FLD NAAF,93904,KFTW,19451130


Result: the catalogue has 3,851 Texas stations in 8 full pages and an empty 9th page, and every one of the
47 ICAO codes is published. Two GHCN ids map to KFTW: the current Meacham Field station (ISD END 2025-08-27)
and an old WBAN of the same airport (ISD END 1945-11-30). We keep the one with the latest ISD row, which leaves 46.

Step 5. Uniqueness checks, then comparison with `config/stations.csv` (the file is not overwritten).

In [6]:
assert final["ghcn_id"].is_unique and final["icao"].is_unique
config = pd.read_csv(ROOT / "config" / "stations.csv", dtype=str)
print("Final stations:", len(final), "| config/stations.csv:", len(config))
print("Same ghcn_id -> icao pairs as the config:",
      set(zip(final["ghcn_id"], final["icao"])) == set(zip(config["ghcn_id"], config["icao"])))
config.head()

Final stations: 46 | config/stations.csv: 46
Same ghcn_id -> icao pairs as the config: True


,ghcn_id,wban,icao,name,lat,lon,forecast_zone,county_zone,timezone
0,USW00003902,03902,KGRK,ROBERT GRAY AAF,31.0667,-97.8333,TXZ158,TXC027,America/Chicago
1,USW00003904,03904,KCLL,COLLEGE STN EASTERWOOD FLD,30.5911,-96.3631,TXZ196,TXC041,America/Chicago
2,USW00003927,03927,KDFW,DAL-FTW WSCMO AP,32.8975,-97.0219,TXZ119,TXC113,America/Chicago
3,USW00003933,03933,KHLR,FT HOOD,31.1333,-97.7167,TXZ158,TXC027,America/Chicago
4,USW00003969,03969,KSEP,STEPHENVILLE CLARK FLD,32.2153,-98.1775,TXZ130,TXC143,America/Chicago


Result: the notebook rebuilds exactly the 46 `ghcn_id`/`icao` pairs of `config/stations.csv`.

### First GHCN query (David's request) and the duplicated-station bug

The first request was made with the config built before the latest-row-per-WBAN rule. We keep
the response in `data/raw/` as evidence.

In [7]:
first = pd.DataFrame(json.loads((RAW / "ghcn_daily_20261116_014223.json").read_text()))
print("Rows:", len(first))
print(first.groupby("STATION").size().to_dict())
print("Duplicated (STATION, DATE):", first.duplicated(["STATION", "DATE"]).sum())

Rows: 1093
{'USW00003902': 363, 'USW00003904': 730}
Duplicated (STATION, DATE): 365


Result: the first query returned USW00003904 twice (730 rows, 365 duplicated station-days). This is how we found
the duplicated WBAN rows in ISD. The same file also shows that USW00003902 has 363 rows, not 365.

## 1. F2 GHCN-Daily profiling (3 stations, 2025, TMAX/TMIN/PRCP)

We request the three stations that are also polled in F3/F4 (KDFW, KIAH, KAUS) for 2025, with
attributes and metric units. The request is made once; if the raw file exists it is reused.

In [8]:
F2_IDS = ["USW00003927", "USW00012960", "USW00013904"]
f2_files = sorted(p for p in RAW.glob("ghcn_daily_3st_2025_*.json") if not p.name.endswith(".headers.json"))
if not f2_files:
    params = {"dataset": "daily-summaries", "stations": ",".join(F2_IDS), "startDate": "2025-01-01",
              "endDate": "2025-12-31", "dataTypes": "TMAX,TMIN,PRCP", "includeAttributes": "true",
              "units": "metric", "format": "json"}
    r = requests.get("https://www.ncei.noaa.gov/access/services/data/v1", params=params, timeout=120)
    r.raise_for_status()
    f2_files = [save_raw(r, RAW / f"ghcn_daily_3st_2025_{utc_stamp()}.json")]
f2_path = f2_files[-1]
f2_raw = json.loads(f2_path.read_text())
print(f2_path.name)
print("First record as received:", f2_raw[0])
gh = pd.DataFrame(f2_raw)
print("Shape:", gh.shape)
print("Python types of TMAX values:", {type(v).__name__ for v in gh["TMAX"].dropna()})

ghcn_daily_3st_2025_20261116_235238.json
First record as received: {'DATE': '2025-01-01', 'STATION': 'USW00003927', 'PRCP_ATTRIBUTES': ',,W', 'TMAX': '10.0', 'TMAX_ATTRIBUTES': ',,W', 'TMIN': '2.2', 'PRCP': '0.0', 'TMIN_ATTRIBUTES': ',,W'}
Shape: (1095, 8)
Python types of TMAX values: {'str'}


Values arrive as strings (`"TMAX": "10.0"`), so we convert them explicitly below.

In [9]:
for col in ["TMAX", "TMIN", "PRCP"]:
    gh[col] = pd.to_numeric(gh[col], errors="coerce")
gh["DATE"] = pd.to_datetime(gh["DATE"])
print("Duplicated (STATION, DATE):", gh.duplicated(["STATION", "DATE"]).sum())

calendar = pd.date_range("2025-01-01", "2025-12-31", freq="D")
full = pd.MultiIndex.from_product([F2_IDS, calendar], names=["STATION", "DATE"]).to_frame(index=False)
check = full.merge(gh[["STATION", "DATE"]].assign(present=True), how="left")
missing = check[check["present"].isna()]
print("Expected station-days:", len(full), "| received:", len(gh), "| missing:", len(missing))
print(missing.groupby("STATION").size().to_dict())

old = first.drop_duplicates(["STATION", "DATE"])
old_dates = set(pd.to_datetime(old.loc[old["STATION"] == "USW00003902", "DATE"]))
print("USW00003902 days missing in the first query:", sorted(d.date().isoformat() for d in set(calendar) - old_dates))

Duplicated (STATION, DATE): 0
Expected station-days: 1095 | received: 1095 | missing: 0
{}
USW00003902 days missing in the first query: ['2025-08-30', '2025-08-31']


Result: no duplicates and no missing days for the three large airports in 2025. Missing days are possible,
though: USW00003902 has no rows for 2025-08-30 and 2025-08-31. They are absent rows, not nulls, so completeness
must be checked against a full calendar.

In [10]:
nulls = (gh[["TMAX", "TMIN", "PRCP"]].isna().mean() * 100).round(2)
print("Null % per element:", nulls.to_dict())
display(gh.groupby("STATION")[["TMAX", "TMIN", "PRCP"]].agg(["min", "max"]))
bad = gh[gh["TMIN"] > gh["TMAX"]]
print("Rows with TMIN > TMAX:", len(bad))

Null % per element: {'TMAX': 0.0, 'TMIN': 0.0, 'PRCP': 0.18}


TMAX        TMIN       PRCP       
             min   max   min   max  min    max
STATION                                       
USW00003927 -2.1  40.0 -11.0  26.7  0.0  100.3
USW00012960  2.8  38.3  -6.6  28.3  0.0   72.1
USW00013904 -1.0  39.4  -8.8  26.1  0.0   83.8

Rows with TMIN > TMAX: 0


Result: only 2 PRCP values are null (0.18%). Ranges are plausible for Texas, and no row has TMIN > TMAX.

In [11]:
flag_counts = {}
for col in ["TMAX", "TMIN", "PRCP"]:
    parts = gh[f"{col}_ATTRIBUTES"].fillna("").str.split(",", expand=True)
    gh[f"{col}_M"], gh[f"{col}_Q"], gh[f"{col}_S"] = parts[0], parts[1], parts[2]
    for kind in ["M", "Q", "S"]:
        flag_counts[(col, kind)] = gh[f"{col}_{kind}"].replace("", "(blank)").value_counts().to_dict()
display(pd.Series(flag_counts).rename("counts").to_frame())
gh["month"] = gh["DATE"].dt.month
display(pd.crosstab(gh["month"], gh["TMAX_S"]))

gh["is_trace"] = gh["PRCP_M"] == "T"
trace = gh[gh["is_trace"]]
print("Trace days (PRCP measurement flag T):", len(trace), "| PRCP values on trace days:", trace["PRCP"].unique().tolist())
print("Example:", trace[["STATION", "DATE", "PRCP", "PRCP_ATTRIBUTES"]].iloc[0].to_dict())

counts
TMAX M           {'(blank)': 1095}
     Q           {'(blank)': 1095}
     S        {'W': 714, '1': 381}
TMIN M           {'(blank)': 1095}
     Q           {'(blank)': 1095}
     S        {'W': 714, '1': 381}
PRCP M  {'(blank)': 962, 'T': 133}
     Q           {'(blank)': 1093}
     S        {'W': 712, '1': 381}

TMAX_S,1,W
month,,
1,0,93
2,0,84
3,0,93
4,0,90
5,0,93
6,0,90
7,0,93
8,15,78
9,90,0


Trace days (PRCP measurement flag T): 133 | PRCP values on trace days: [0.0]
Example: {'STATION': 'USW00003927', 'DATE': Timestamp('2025-01-05 00:00:00'), 'PRCP': 0.0, 'PRCP_ATTRIBUTES': 'T,,W'}


Result: quality flags are blank, so no value failed the NCEI checks. Source flags mix `W` (WBAN/ASOS summary
of the day from ISD) and `1`. The crosstab shows that January to most of August 2025 have `W`, and late August to
December have `1`. We do not have documentation for `1`, so we only record that recent months come from a
different source stream than older months. Values can be re-published later, so F2 is not frozen. There are 133 trace days with
`PRCP = 0.0` and flag `T`. Decision: we keep 0.0 and add `is_trace`, so wet-day statistics can include or exclude them.

## 2. F1 NOAA Storm Events (bulk CSV.gz)

We read the bulk listing, take the latest `d2025` details file, download it once to `data/raw/`,
and keep Texas rows with `STATE_FIPS == "48"`.

In [12]:
BULK = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
listing_files = sorted(RAW.glob("storm_events_listing_*.html"))
if not listing_files:
    r = requests.get(BULK, timeout=60)
    r.raise_for_status()
    listing_files = [save_raw(r, RAW / f"storm_events_listing_{utc_stamp()}.html")]
listing = listing_files[-1].read_text()
rows = re.findall(r'href="(StormEvents_details-ftp_v1\.0_d(\d{4})_c(\d{8})\.csv\.gz)".*?<td align="right">([^<]+)</td>\s*<td align="right">(\d+)</td>',
                  listing, flags=re.S)
details = pd.DataFrame(rows, columns=["file", "year", "compiled", "modified", "bytes"]).astype({"year": int, "bytes": int})
print("Details files in listing:", len(details), "| years:", details["year"].min(), "-", details["year"].max())
f1_name = details[details["year"] == 2025].sort_values("compiled")["file"].iloc[-1]
print("Latest 2025 details file:", f1_name)

f1_path = RAW / f1_name
if not f1_path.exists():
    r = requests.get(BULK + f1_name, timeout=300)
    r.raise_for_status()
    save_raw(r, f1_path)
se_all = pd.read_csv(f1_path, dtype=str, compression="gzip")
se = se_all[se_all["STATE_FIPS"] == "48"].copy()
print("US rows:", len(se_all), "| Texas rows:", len(se), "| fields:", se.shape[1])
print("Rows with STATE == 'TEXAS':", (se_all["STATE"] == "TEXAS").sum())
tx_sample = SAMPLES / f1_name.replace("StormEvents_details-ftp_v1.0", "storm_events_tx").replace(".csv.gz", ".csv")
se.to_csv(tx_sample, index=False)
print("Texas sample saved to", tx_sample.relative_to(ROOT))

Details files in listing: 77 | years: 1950 - 2026
Latest 2025 details file: StormEvents_details-ftp_v1.0_d2025_c20260819.csv.gz


US rows: 72360 | Texas rows: 5641 | fields: 51
Rows with STATE == 'TEXAS': 5641
Texas sample saved to workshop_1/samples/storm_events_tx_d2025_c20260819.csv


Result: the 2025 file has 72,360 US rows and 5,641 Texas rows. Filtering by `STATE_FIPS == "48"` and by
`STATE == "TEXAS"` gives the same count here, but we keep the FIPS rule because it does not depend on text.

In [13]:
print("Duplicated EVENT_ID:", se["EVENT_ID"].duplicated().sum())
print("CZ_TYPE:", se["CZ_TYPE"].value_counts().to_dict())
print("CZ_TIMEZONE:", se["CZ_TIMEZONE"].value_counts().to_dict())
ex = se.iloc[0]
print("Date format example:", ex["BEGIN_DATE_TIME"], "| BEGIN_YEARMONTH", ex["BEGIN_YEARMONTH"], "BEGIN_DAY", ex["BEGIN_DAY"], "BEGIN_TIME", ex["BEGIN_TIME"])
el_paso = se[se["CZ_TIMEZONE"].str.startswith("MST")]
print("MST example:", el_paso[["EVENT_ID", "CZ_NAME", "BEGIN_DATE_TIME", "CZ_TIMEZONE"]].iloc[0].to_dict() if len(el_paso) else "none")
lat_missing = se["BEGIN_LAT"].isna()
print("Missing BEGIN_LAT by CZ_TYPE (share):", lat_missing.groupby(se["CZ_TYPE"]).mean().round(3).to_dict())
print("Missing BEGIN_LAT by CZ_TYPE (rows):", lat_missing.groupby(se["CZ_TYPE"]).sum().to_dict())

Duplicated EVENT_ID: 0
CZ_TYPE: {'C': 3572, 'Z': 2069}
CZ_TIMEZONE: {'CST-6': 5530, 'MST-7': 111}
Date format example: 09-JAN-25 20:00:00 | BEGIN_YEARMONTH 202501 BEGIN_DAY 9 BEGIN_TIME 2000
MST example: {'EVENT_ID': '1305042', 'CZ_NAME': 'SOUTHERN HUDSPETH HIGHLANDS', 'BEGIN_DATE_TIME': '01-DEC-25 00:00:00', 'CZ_TIMEZONE': 'MST-7'}
Missing BEGIN_LAT by CZ_TYPE (share): {'C': 0.0, 'Z': 1.0}
Missing BEGIN_LAT by CZ_TYPE (rows): {'C': 0, 'Z': 2069}


Result: EVENT_ID is unique. Times are local standard time with a separate offset (`CST-6`, and `MST-7` for
111 rows in far West Texas), and `BEGIN_DATE_TIME` has a 2-digit year (`09-JAN-25 20:00:00`). All 2,069 zone rows
(`CZ_TYPE=Z`) have no coordinates, and all county rows have them.

In [14]:
MULT = {"K": 1e3, "M": 1e6, "B": 1e9}


def parse_damage(text):
    """'10.00K' -> 10000.0, '0.00K' -> 0.0, blank -> NaN (unknown, not zero)."""
    if pd.isna(text) or str(text).strip() == "":
        return float("nan")
    text = str(text).strip().upper()
    if text[-1] in MULT:
        return float(text[:-1] or 0) * MULT[text[-1]]
    return float(text)


se["damage_property_usd"] = se["DAMAGE_PROPERTY"].map(parse_damage)
print("DAMAGE_PROPERTY examples:", se["DAMAGE_PROPERTY"].dropna().drop_duplicates().head(8).tolist())
print("Suffixes used:", se["DAMAGE_PROPERTY"].dropna().str[-1].value_counts().to_dict())
print(f"Blank (unknown): {se['damage_property_usd'].isna().mean():.1%} | zero: {(se['damage_property_usd'] == 0).mean():.1%}"
      f" | max: {se['damage_property_usd'].max():,.0f} USD")

se["MAG"] = pd.to_numeric(se["MAGNITUDE"], errors="coerce")
mag = se.groupby("EVENT_TYPE").agg(rows=("EVENT_ID", "size"), with_magnitude=("MAG", "count"),
                                   mag_min=("MAG", "min"), mag_max=("MAG", "max"),
                                   magnitude_types=("MAGNITUDE_TYPE", lambda s: ",".join(sorted(s.dropna().unique()))))
mag.sort_values("rows", ascending=False)

DAMAGE_PROPERTY examples: ['0.00K', '10.00K', '50.00K', '100.00K', '1.00K', '25.00K', '3.00K', '7.00K']
Suffixes used: {'K': 4735, 'M': 20}
Blank (unknown): 15.7% | zero: 69.8% | max: 200,000,000 USD


,rows,with_magnitude,mag_min,mag_max,magnitude_types
EVENT_TYPE,,,,,
Hail,1453,1453,0.75,6.0,
Thunderstorm Wind,1205,1205,35.00,104.0,"EG,MG"
Flash Flood,624,0,NaN,NaN,
Drought,611,0,NaN,NaN,
High Wind,448,448,38.00,83.0,"EG,MG,MS"
Heat,347,0,NaN,NaN,
Cold/Wind Chill,188,0,NaN,NaN,
Tornado,143,0,NaN,NaN,
Winter Weather,131,0,NaN,NaN,


Result: damage is text with K or M suffix, 15.7% blank. Blank means unknown, so it is not converted to 0.
MAGNITUDE is only filled for hail (inches), thunderstorm, high and strong wind (knots, `EG` estimated or `MG`
measured). It is empty for heat, flood, tornado and cold events, so severity needs a different field per type.

## 3. F3 NWS alerts, poll 1 (2026-11-16 23:12:34 UTC)

In [15]:
POLL1 = SAMPLES / "poll1"
POLL2 = SAMPLES / "poll2"


def load_alerts(folder, kind):
    path = sorted(folder.glob(f"nws_alerts_{kind}_*.geojson"))[-1]
    features = json.loads(path.read_text())["features"]
    rows = []
    for f in features:
        p = f["properties"]
        rows.append({"id": p["id"], "event": p["event"], "messageType": p["messageType"], "severity": p["severity"],
                     "status": p["status"], "sent": p["sent"], "expires": p["expires"],
                     "n_references": len(p.get("references") or []), "has_geometry": f.get("geometry") is not None,
                     "ugc": p["geocode"].get("UGC", []), "areaDesc": p["areaDesc"]})
    return pd.DataFrame(rows)


def load_observations(folder):
    rows = []
    for path in sorted(folder.glob("nws_observations_*.geojson")):
        if path.name.endswith(".headers.json"):
            continue
        for f in json.loads(path.read_text())["features"]:
            p = f["properties"]
            row = {"station": p["stationId"], "timestamp": p["timestamp"], "lon": f["geometry"]["coordinates"][0],
                   "lat": f["geometry"]["coordinates"][1]}
            for key, val in p.items():
                if isinstance(val, dict) and "unitCode" in val:
                    row[f"{key}.value"] = val.get("value")
                    row[f"{key}.unit"] = val.get("unitCode")
                    row[f"{key}.qc"] = val.get("qualityControl")
            rows.append(row)
    return pd.DataFrame(rows)

In [16]:
al = load_alerts(POLL1, "recent")
act = load_alerts(POLL1, "active")
print("Recent alerts (alerts?area=TX&limit=500):", len(al), "| active now:", len(act))
print("Duplicated id:", al["id"].duplicated().sum())
print("UTC offsets in 'sent':", al["sent"].str[-6:].value_counts().to_dict())
print("Example sent:", al["sent"].iloc[0])
print(f"Alerts without geometry: {(~al['has_geometry']).mean():.1%} ({(~al['has_geometry']).sum()} of {len(al)})")
print("messageType:", al["messageType"].value_counts().to_dict())
print("severity:", al["severity"].value_counts().to_dict())
print("Alerts with references:", (al["n_references"] > 0).sum())
print("sent range:", al["sent"].min(), "to", al["sent"].max())
al["event"].value_counts().to_frame("alerts")

Recent alerts (alerts?area=TX&limit=500): 172 | active now: 6
Duplicated id: 0
UTC offsets in 'sent': {'-06:00': 166, '-07:00': 6}
Example sent: 2026-11-16T16:49:00-06:00
Alerts without geometry: 44.8% (77 of 172)
messageType: {'Alert': 85, 'Update': 67, 'Cancel': 20}
severity: {'Moderate': 79, 'Minor': 61, 'Severe': 32}
Alerts with references: 87
sent range: 2026-11-09T18:42:00-06:00 to 2026-11-16T16:49:00-06:00


,alerts
event,
Flood Warning,31
Dense Fog Advisory,31
Flood Advisory,29
Special Weather Statement,26
Wind Advisory,16
Severe Thunderstorm Warning,11
Coastal Flood Advisory,10
Rip Current Statement,9
Beach Hazards Statement,5


Result: 172 alerts in the last 7 days, 6 active at poll time, no duplicated ids. `sent` uses local offsets
(-06:00 and -07:00), not UTC. 44.8% of alerts have no geometry and can only be located by UGC zone codes.
There are 85 Alert, 67 Update and 20 Cancel messages, and 87 messages reference earlier ids, so one real
warning is a chain of messages. Poll 1 returned 172 recent alerts and 6 active alerts, which is the complete
set of Texas alerts available in the NWS 7-day window: the response is below `limit=500` and its
`pagination.next` link returns an empty page (checked at 2026-11-17 00:19 UTC, saved in
`workshop_1/samples/poll2/`). So for our scope F3 is profiled on the full source, which the rubric allows
when the source has fewer than 200 records.

## 4. F4 NWS station observations, poll 1

In [17]:
ob = load_observations(POLL1)
ob["ts"] = pd.to_datetime(ob["timestamp"], utc=True)
span = ob.groupby("station").agg(records=("ts", "size"), first=("ts", "min"), last=("ts", "max"))
span["hours_covered"] = ((span["last"] - span["first"]).dt.total_seconds() / 3600).round(1)
display(span)
print("Duplicated (station, timestamp):", ob.duplicated(["station", "timestamp"]).sum())
print("Example coordinates [lon, lat]:", [ob["lon"].iloc[0], ob["lat"].iloc[0]])
value_cols = [c for c in ob.columns if c.endswith(".value")]
null_pct = (ob[value_cols].isna().mean() * 100).round(1).sort_values()
display(null_pct.rename("null_pct").to_frame())

,records,first,last,hours_covered
station,,,,
KAUS,500,2026-11-13 00:20:00+00:00,2026-11-16 22:55:00+00:00,94.6
KDFW,500,2026-11-12 21:40:00+00:00,2026-11-16 22:55:00+00:00,97.2
KIAH,500,2026-11-13 06:25:00+00:00,2026-11-16 22:55:00+00:00,88.5


Duplicated (station, timestamp): 0
Example coordinates [lon, lat]: [np.float64(-97.67), np.float64(30.19)]


,null_pct
elevation.value,0.0
temperature.value,0.1
dewpoint.value,0.1
relativeHumidity.value,0.1
barometricPressure.value,0.7
windSpeed.value,12.2
windDirection.value,12.4
visibility.value,14.7
windChill.value,82.5
seaLevelPressure.value,84.0


Result: each station returned exactly 500 records covering 88.5 to 97.2 hours, not the full 7 days. The default
request is capped, so the incremental load must request by time window. No duplicates on (station, timestamp).
Core variables are almost complete (temperature 0.1% null), while gust, heat index and precipitation are
mostly null because they are only reported when they apply.

In [18]:
units = {c[:-5]: sorted(ob[c].dropna().unique()) for c in ob.columns if c.endswith(".unit")}
display(pd.Series(units).rename("unitCode").to_frame())
print("qualityControl codes for temperature:", ob["temperature.qc"].value_counts().to_dict())
print("Temperature range (degC):", ob["temperature.value"].min(), "to", ob["temperature.value"].max())
print("Example null value:", ob.loc[ob["windGust.value"].isna(), ["station", "timestamp", "windGust.value", "windGust.unit", "windGust.qc"]].iloc[0].to_dict())

,unitCode
elevation,[wmoUnit:m]
temperature,[wmoUnit:degC]
dewpoint,[wmoUnit:degC]
windDirection,[wmoUnit:degree_(angle)]
windSpeed,[wmoUnit:km_h-1]
windGust,[wmoUnit:km_h-1]
barometricPressure,[wmoUnit:Pa]
seaLevelPressure,[wmoUnit:Pa]
visibility,[wmoUnit:m]
maxTemperatureLast24Hours,[wmoUnit:degC]


qualityControl codes for temperature: {'V': 1498, 'Z': 2}
Temperature range (degC): 7.0 to 27.2
Example null value: {'station': 'KAUS', 'timestamp': '2026-11-16T22:55:00+00:00', 'windGust.value': nan, 'windGust.unit': 'wmoUnit:km_h-1', 'windGust.qc': 'Z'}


Result: every variable is an object with value, WMO unit code and QC code. Units differ from F2 in wind
(km/h) and pressure (Pa). Temperature is in degC like F2. A null keeps its unit and has QC code `Z`.

## 5. Cross-source overlap

F2 and F4 are the same physical station by construction (the crosswalk of section 0 links each
GHCN id to its ICAO code). For F1 and F2 we look for candidate pairs: an F1 event in the county of
the station (county row with the station county FIPS, or zone row with the station forecast zone),
on the same local day as the F2 record, with a compatible variable.

In [19]:
VAR = {"Heat": "TMAX", "Excessive Heat": "TMAX", "Flash Flood": "PRCP", "Flood": "PRCP", "Heavy Rain": "PRCP",
       "Thunderstorm Wind": "PRCP", "Hail": "PRCP", "Tornado": "PRCP", "Cold/Wind Chill": "TMIN",
       "Extreme Cold/Wind Chill": "TMIN", "Frost/Freeze": "TMIN", "Winter Storm": "TMIN", "Winter Weather": "TMIN"}
st = config[config["ghcn_id"].isin(F2_IDS)].copy()
st["county_fips"] = st["county_zone"].str[-3:].astype(int)
st["zone_num"] = st["forecast_zone"].str[-3:].astype(int)
se["local_date"] = pd.to_datetime(se["BEGIN_YEARMONTH"] + se["BEGIN_DAY"].str.zfill(2), format="%Y%m%d")
se["cz_fips"] = se["CZ_FIPS"].astype(int)

pairs = []
for _, s in st.iterrows():
    in_area = ((se["CZ_TYPE"] == "C") & (se["cz_fips"] == s["county_fips"])) | \
              ((se["CZ_TYPE"] == "Z") & (se["cz_fips"] == s["zone_num"]))
    cand = se[in_area & se["EVENT_TYPE"].isin(VAR)].copy()
    cand["STATION"], cand["icao"] = s["ghcn_id"], s["icao"]
    pairs.append(cand.merge(gh, left_on=["STATION", "local_date"], right_on=["STATION", "DATE"]))
pairs = pd.concat(pairs, ignore_index=True)
pairs["f2_element"] = pairs["EVENT_TYPE"].map(VAR)
pairs["f2_value"] = [row[row["f2_element"]] for _, row in pairs.iterrows()]
print("Candidate pairs:", len(pairs))
print(pairs.groupby(["icao", "EVENT_TYPE"]).size().to_dict())
cols = ["EVENT_ID", "EVENT_TYPE", "CZ_TYPE", "CZ_NAME", "BEGIN_DATE_TIME", "CZ_TIMEZONE", "MAGNITUDE",
        "icao", "STATION", "DATE", "f2_element", "f2_value"]
show = pairs.sort_values(["f2_element", "f2_value"], ascending=False).groupby("f2_element").head(3)[cols]
show

Candidate pairs: 143
{('KAUS', 'Flash Flood'): 7, ('KAUS', 'Hail'): 15, ('KAUS', 'Thunderstorm Wind'): 6, ('KAUS', 'Winter Storm'): 1, ('KAUS', 'Winter Weather'): 1, ('KDFW', 'Cold/Wind Chill'): 2, ('KDFW', 'Extreme Cold/Wind Chill'): 1, ('KDFW', 'Flash Flood'): 44, ('KDFW', 'Hail'): 16, ('KDFW', 'Heat'): 4, ('KDFW', 'Thunderstorm Wind'): 16, ('KDFW', 'Tornado'): 1, ('KDFW', 'Winter Storm'): 1, ('KIAH', 'Flash Flood'): 2, ('KIAH', 'Flood'): 1, ('KIAH', 'Hail'): 8, ('KIAH', 'Thunderstorm Wind'): 14, ('KIAH', 'Tornado'): 2, ('KIAH', 'Winter Storm'): 1}


,EVENT_ID,EVENT_TYPE,CZ_TYPE,CZ_NAME,BEGIN_DATE_TIME,CZ_TIMEZONE,MAGNITUDE,icao,STATION,DATE,f2_element,f2_value
8,1231214,Winter Storm,Z,DALLAS,09-JAN-25 10:00:00,CST-6,NaN,KDFW,USW00003927,2025-01-09,TMIN,-0.5
121,1234272,Winter Weather,Z,TRAVIS,22-FEB-25 02:00:00,CST-6,NaN,KAUS,USW00013904,2025-02-22,TMIN,-1.0
85,1231497,Winter Storm,Z,INLAND HARRIS,20-JAN-25 18:00:00,CST-6,NaN,KIAH,USW00012960,2025-01-20,TMIN,-2.1
56,1287952,Heat,Z,DALLAS,24-JUL-25 11:00:00,CST-6,NaN,KDFW,USW00003927,2025-07-24,TMAX,36.7
68,1292129,Heat,Z,DALLAS,15-AUG-25 11:00:00,CST-6,NaN,KDFW,USW00003927,2025-08-15,TMAX,36.7
52,1287970,Heat,Z,DALLAS,28-JUL-25 11:00:00,CST-6,NaN,KDFW,USW00003927,2025-07-28,TMAX,36.1
69,1300291,Flash Flood,C,DALLAS,20-NOV-25 05:30:00,CST-6,NaN,KDFW,USW00003927,2025-11-20,PRCP,100.3
70,1300295,Flash Flood,C,DALLAS,20-NOV-25 12:15:00,CST-6,NaN,KDFW,USW00003927,2025-11-20,PRCP,100.3
71,1300296,Flash Flood,C,DALLAS,20-NOV-25 06:46:00,CST-6,NaN,KDFW,USW00003927,2025-11-20,PRCP,100.3


Result: 143 candidate pairs for the three stations in 2025. Examples: the Dallas winter storm on 2025-01-09
with TMIN -0.5 at KDFW, Dallas heat events with TMAX 36.7, and three Flash Flood reports on 2025-11-20
with PRCP 100.3 mm at KDFW. What differs: F1 is a report for a county or zone with local time, F2 is one
value per station-day, several F1 events can match one F2 day, and a hail or wind report only says that a
storm happened, not that the station measured it. These pairs are candidates for the indicator validation,
not confirmed matches.

## 6. Data-quality problems

Each problem has its extent and one example record taken from the outputs above.

In [20]:
issues = []


def add(source, problem, extent, example, handling):
    issues.append({"source": source, "problem": problem, "extent": extent, "example": example, "handling": handling})


add("F2", "Numeric values arrive as JSON strings", "all values",
    f'{f2_raw[0]["STATION"]} {f2_raw[0]["DATE"]}: "TMAX": "{f2_raw[0]["TMAX"]}"', "pd.to_numeric(errors='coerce')")
add("F2", "Missing station-days are absent rows, not nulls", f"{len(missing)} missing of {len(full)} expected (3 stations, 2025); USW00003902 misses 2 days in the first query",
    "USW00003902: no rows for 2025-08-30 and 2025-08-31", "check completeness against a full calendar")
t = trace.iloc[0]
add("F2", "Trace precipitation stored as 0.0 with measurement flag T", f"{len(trace)} trace days of {len(gh)} rows",
    f'{t["STATION"]} {t["DATE"].date()}: PRCP={t["PRCP"]}, PRCP_ATTRIBUTES="{t["PRCP_ATTRIBUTES"]}"', "keep 0.0 and add is_trace flag")
add("F2", "Duplicated station-days after a bad crosswalk (same WBAN twice)", f"{first.duplicated(['STATION', 'DATE']).sum()} duplicated station-days in the first query",
    f"USW00003904: {(first['STATION'] == 'USW00003904').sum()} rows for 2025", "latest ISD row per WBAN, assert unique ids")
add("ISD", "Two GHCN ids map to the same ICAO code", f"{len(shared)} GHCN ids share {shared['icao'].nunique()} ICAO code",
    "; ".join(f"{r.id} {r['name']} WBAN {r.wban} END {r.END}" for _, r in shared.iterrows()), "keep the id with the latest ISD END date")
add("ISD", "Sentinel 99999 for missing WBAN (999999 for USAF)", f"{isd['WBAN'].isna().sum()} ISD rows",
    "isd-history.csv row 2: USAF=007018, WBAN=99999", "convert sentinels to NULL")
ex = se.iloc[0]
add("F1", "Local standard time with offset field and 2-digit year", "all rows",
    f'EVENT_ID {ex["EVENT_ID"]}: BEGIN_DATE_TIME="{ex["BEGIN_DATE_TIME"]}", CZ_TIMEZONE="{ex["CZ_TIMEZONE"]}"',
    "build the timestamp from BEGIN_YEARMONTH, BEGIN_DAY, BEGIN_TIME and apply the offset")
d = se[se["DAMAGE_PROPERTY"].fillna("").str.contains("[KMB]$") & (se["damage_property_usd"] > 0)].iloc[0]
add("F1", "Damage as text with K/M/B suffix; blank means unknown", f"{se['DAMAGE_PROPERTY'].isna().mean():.1%} blank",
    f'EVENT_ID {d["EVENT_ID"]}: DAMAGE_PROPERTY="{d["DAMAGE_PROPERTY"]}"', "parse to USD, keep blank as NULL")
h = se[(se["EVENT_TYPE"] == "Hail") & se["MAG"].notna()].iloc[0]
w = se[(se["EVENT_TYPE"] == "Thunderstorm Wind") & se["MAG"].notna()].iloc[0]
add("F1", "MAGNITUDE units depend on EVENT_TYPE", "hail in inches, wind in knots, empty for heat and flood",
    f'EVENT_ID {h["EVENT_ID"]} Hail MAGNITUDE={h["MAGNITUDE"]}; EVENT_ID {w["EVENT_ID"]} Thunderstorm Wind MAGNITUDE={w["MAGNITUDE"]} {w["MAGNITUDE_TYPE"]}',
    "unit column derived from EVENT_TYPE")
z = se[(se["CZ_TYPE"] == "Z") & se["BEGIN_LAT"].isna()].iloc[0]
add("F1", "Zone-based rows have no coordinates", f"{se.loc[se['CZ_TYPE'] == 'Z', 'BEGIN_LAT'].isna().mean():.1%} of CZ_TYPE=Z rows",
    f'EVENT_ID {z["EVENT_ID"]}: {z["EVENT_TYPE"]}, CZ_TYPE=Z, CZ_NAME={z["CZ_NAME"]}, BEGIN_LAT empty', "locate by zone, latitude NULL")
a = al.iloc[0]
add("F3", "Timestamps carry local offsets, not UTC", f"offsets {al['sent'].str[-6:].value_counts().to_dict()}",
    f'{a["id"][-40:]}: sent="{a["sent"]}"', "parse with offset, store UTC")
g = al[~al["has_geometry"]].iloc[0] if (~al["has_geometry"]).any() else al.iloc[0]
add("F3", "Alerts without geometry (zone list only)", f"{(~al['has_geometry']).sum()} of {len(al)} alerts",
    f'{g["event"]}, UGC={g["ugc"][:3]}', "locate by UGC zone codes")
o = ob.iloc[0]
add("F3/F4", "GeoJSON coordinates are [lon, lat], opposite order to F2", "all features",
    f'{o["station"]} {o["timestamp"]}: coordinates=[{o["lon"]}, {o["lat"]}]', "swap when mapping")
n = ob.loc[ob["windGust.value"].isna()].iloc[0]
add("F4", "Values are {value, unitCode, qualityControl} objects; nulls are value: null",
    f"windGust null {ob['windGust.value'].isna().mean():.1%}", f'{n["station"]} {n["timestamp"]}: windGust.value=None, unitCode={n["windGust.unit"]}',
    "flatten, keep unit and QC code")
add("F4", "Default response capped at 500 records, less than the 7 days kept by NWS", f"500 records per station, {span['hours_covered'].min()} to {span['hours_covered'].max()} hours",
    f'KDFW first={span.loc["KDFW", "first"]}, last={span.loc["KDFW", "last"]}', "request by start/end window, poll every hour")

dq = pd.DataFrame(issues)
dq.to_csv(ROOT / "workshop_1" / "data_quality_issues.csv", index=False)
print("Problems:", len(dq))
dq

Problems: 15


,source,problem,extent,example,handling
0,F2,Numeric values arrive as JSON strings,all values,"USW00003927 2025-01-01: ""TMAX"": ""10.0""",pd.to_numeric(errors='coerce')
1,F2,"Missing station-days are absent rows, not nulls","0 missing of 1095 expected (3 stations, 2025);...",USW00003902: no rows for 2025-08-30 and 2025-0...,check completeness against a full calendar
2,F2,Trace precipitation stored as 0.0 with measure...,133 trace days of 1095 rows,"USW00003927 2025-01-05: PRCP=0.0, PRCP_ATTRIBU...",keep 0.0 and add is_trace flag
3,F2,Duplicated station-days after a bad crosswalk ...,365 duplicated station-days in the first query,USW00003904: 730 rows for 2025,"latest ISD row per WBAN, assert unique ids"
4,ISD,Two GHCN ids map to the same ICAO code,2 GHCN ids share 1 ICAO code,USW00013961 FT WORTH MEACHAM FLD WBAN 13961 EN...,keep the id with the latest ISD END date
5,ISD,Sentinel 99999 for missing WBAN (999999 for USAF),25367 ISD rows,"isd-history.csv row 2: USAF=007018, WBAN=99999",convert sentinels to NULL
6,F1,Local standard time with offset field and 2-di...,all rows,"EVENT_ID 1230373: BEGIN_DATE_TIME=""09-JAN-25 2...","build the timestamp from BEGIN_YEARMONTH, BEGI..."
7,F1,Damage as text with K/M/B suffix; blank means ...,15.7% blank,"EVENT_ID 1256818: DAMAGE_PROPERTY=""10.00K""","parse to USD, keep blank as NULL"
8,F1,MAGNITUDE units depend on EVENT_TYPE,"hail in inches, wind in knots, empty for heat ...",EVENT_ID 1230425 Hail MAGNITUDE=1.00; EVENT_ID...,unit column derived from EVENT_TYPE
9,F1,Zone-based rows have no coordinates,100.0% of CZ_TYPE=Z rows,"EVENT_ID 1230373: Coastal Flood, CZ_TYPE=Z, CZ...","locate by zone, latitude NULL"


The table is saved to `workshop_1/data_quality_issues.csv`.

## 7. Poll 2 and comparison with poll 1

Poll 2 was taken with `workshop_1/poll_nws.py` into `workshop_1/samples/poll2/`, using the same
requests as poll 1.

In [21]:
def poll_time(folder):
    h = json.loads(sorted(folder.glob("nws_alerts_active_*.headers.json"))[-1].read_text())
    return h["retrieved_utc"], h["headers"].get("ETag"), h["headers"].get("Cache-Control")


if not POLL2.exists() or not list(POLL2.glob("*.geojson")):
    print("Poll 2 is not available.")
else:
    print("poll 1:", poll_time(POLL1))
    print("poll 2:", poll_time(POLL2))
    act2 = load_alerts(POLL2, "active")
    rec2 = load_alerts(POLL2, "recent")
    s1, s2 = set(act["id"]), set(act2["id"])
    print(f"Active alerts: poll1={len(s1)} poll2={len(s2)} new={len(s2 - s1)} no longer active={len(s1 - s2)} kept={len(s1 & s2)}")
    r1, r2 = set(al["id"]), set(rec2["id"])
    print(f"Recent alerts: poll1={len(r1)} poll2={len(r2)} new={len(r2 - r1)} left the window={len(r1 - r2)}")
    new_alerts = rec2[rec2["id"].isin(r2 - r1)]
    print("New alerts by messageType:", new_alerts["messageType"].value_counts().to_dict())
    print("New alerts with references to earlier ids:", (new_alerts["n_references"] > 0).sum())
    display(new_alerts[["event", "messageType", "sent", "n_references"]].head(10))

    ob2 = load_observations(POLL2)
    ob2["ts"] = pd.to_datetime(ob2["timestamp"], utc=True)
    rows = []
    for station in sorted(ob["station"].unique()):
        t1 = set(ob.loc[ob["station"] == station, "ts"])
        t2 = set(ob2.loc[ob2["station"] == station, "ts"])
        rows.append({"station": station, "poll1_records": len(t1), "poll2_records": len(t2),
                     "new_timestamps": len(t2 - t1), "dropped_oldest": len(t1 - t2),
                     "poll1_latest": max(t1), "poll2_latest": max(t2)})
    obs_cmp = pd.DataFrame(rows)
    display(obs_cmp)

poll 1: ('20261116_231234', 'W/"0f24a6305e212bc173099345d62abd84"', 'public, max-age=4, s-maxage=5')
poll 2: ('20261117_001350', 'W/"a4c902ce4f32a804f5f9a27f4b881ac1"', 'public, max-age=5, s-maxage=5')
Active alerts: poll1=6 poll2=6 new=0 no longer active=0 kept=6
Recent alerts: poll1=172 poll2=172 new=0 left the window=0
New alerts by messageType: {}
New alerts with references to earlier ids: 0


,event,messageType,sent,n_references


,station,poll1_records,poll2_records,new_timestamps,dropped_oldest,poll1_latest,poll2_latest
0,KAUS,500,500,4,4,2026-11-16 22:55:00+00:00,2026-11-16 23:55:00+00:00
1,KDFW,500,501,4,3,2026-11-16 22:55:00+00:00,2026-11-16 23:55:00+00:00
2,KIAH,500,500,4,4,2026-11-16 22:55:00+00:00,2026-11-16 23:55:00+00:00


Result: poll 2 was taken at 2026-11-17 00:13:10 UTC, about one hour after poll 1. The alert ids did not
change (6 active, 172 recent, none new and none removed), but the ETag of the active list changed, so the
ETag alone does not mean new alerts and we must compare ids. For F4 each station has 3 or 4 new timestamps,
and the same number of old records left the 500-record window. The latest report moved from 22:55 to
23:55 UTC at KAUS and KIAH and to 23:40 UTC at KDFW. The poll-1 alerts already show the update model:
Update and Cancel are new messages with their own id and point to earlier ids in `references`.

## 8. Volume estimate and partitioning

In [22]:
f2_years = 2026 - 1991 + 1
f2_rows = len(config) * f2_years * 365
print(f"F2: {len(config)} stations x {f2_years} years x 365 days = {f2_rows:,} station-days, "
      f"x 3 elements = {f2_rows * 3:,} values")
bytes_per_station_year = f2_path.stat().st_size / len(F2_IDS)
print(f"F2 JSON size: about {bytes_per_station_year / 1e6:.2f} MB per station-year, "
      f"about {bytes_per_station_year * len(config) * f2_years / 1e6:,.0f} MB in total, {len(config) * f2_years:,} requests")

f1 = details[details["year"] >= 1996]
print(f"F1: {len(f1)} yearly details files from 1996 to {f1['year'].max()}, {f1['bytes'].sum() / 1e6:,.0f} MB compressed (whole US)")
print(f"F1 2025: {details.loc[details['file'] == f1_name, 'bytes'].iloc[0] / 1e6:.1f} MB compressed, {len(se_all):,} US rows, {len(se):,} Texas rows")
display(f1[["year", "compiled", "bytes"]].assign(MB=lambda d: (d["bytes"] / 1e6).round(1)).drop(columns="bytes").set_index("year").T)

obs_per_hour = (span["records"] / span["hours_covered"]).mean()
print(f"F4: about {obs_per_hour:.1f} observations per station per hour, about {obs_per_hour * 24 * 365 * len(config):,.0f} per year for {len(config)} stations")
print(f"F3: {len(al)} Texas alerts returned by the recent endpoint in poll 1 (sent from {al['sent'].min()} to {al['sent'].max()})")

F2: 46 stations x 36 years x 365 days = 604,440 station-days, x 3 elements = 1,813,320 values
F2 JSON size: about 0.06 MB per station-year, about 99 MB in total, 1,656 requests
F1: 31 yearly details files from 1996 to 2026, 313 MB compressed (whole US)
F1 2025: 12.6 MB compressed, 72,360 US rows, 5,641 Texas rows


year,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
compiled,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,20260323,...,20261026,20260323,20260323,20260323,20260323,20260519,20260323,20260323,20260323,20260323,20260625,20260323,20260728,20260819,20261026
MB,6.3,6.6,10.3,10.4,7.9,6.9,7.2,7.2,7.3,7.8,7.6,9.8,12.3,10.3,11.7,...,14.8,11.7,11.1,10.0,9.1,9.3,9.6,11.6,10.4,10.6,11.8,12.9,12.7,12.6,9.2


F4: about 5.4 observations per station per hour, about 2,159,753 per year for 46 stations
F3: 172 Texas alerts returned by the recent endpoint in poll 1 (sent from 2026-11-09T18:42:00-06:00 to 2026-11-16T16:49:00-06:00)
